In [1]:
# Load the landmarks CSV and add background type information
import os
import pandas as pd

dataset_path = "../dataset"

# Load the extracted landmarks
df_clean = pd.read_csv("../nsl_landmarks_v2.csv")

# Count images in Plain Background to determine the split
plain_count = 0
for label in os.listdir(os.path.join(dataset_path, "Plain Background")):
    class_path = os.path.join(dataset_path, "Plain Background", label)
    if os.path.isdir(class_path):
        plain_count += sum(1 for f in os.listdir(class_path) if f.lower().endswith((".jpg", ".jpeg", ".png")))

# Add bg_type column based on the order of processing in the extraction script
# Plain Background is processed first, then Random Background
df_clean["bg_type"] = ["Plain Background" if i < plain_count else "Random Background" for i in range(len(df_clean))]


In [2]:
# Filter by background
plain_df = df_clean[df_clean["bg_type"] == "Plain Background"]
random_df = df_clean[df_clean["bg_type"] == "Random Background"]

print("Plain Background samples:", len(plain_df))
print("Random Background samples:", len(random_df))

Plain Background samples: 36000
Random Background samples: 12474


In [3]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

# Features & labels
X_train = plain_df.drop(["label", "bg_type"], axis=1).values
y_train = plain_df["label"].values

X_test = random_df.drop(["label", "bg_type"], axis=1).values
y_test = random_df["label"].values

# Train model
model_cbt = MLPClassifier(hidden_layer_sizes=(128,64), activation='relu', max_iter=300)
model_cbt.fit(X_train, y_train)

# Evaluate
y_pred = model_cbt.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Cross-background Test Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Cross-background Test Accuracy: 0.922639089305756

Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.00      0.00      0.00         0
           2       0.76      0.81      0.79       467
           3       0.93      0.99      0.96       477
           4       0.99      0.90      0.95       460
           5       0.99      1.00      0.99       498
           6       0.96      0.99      0.97       499
           7       0.95      0.96      0.95       323
           8       0.97      0.90      0.93       350
           9       0.97      1.00      0.98       499
          10       0.00      0.00      0.00         0
          12       0.00      0.00      0.00         0
          13       0.92      1.00      0.96        47
          14       1.00      0.56      0.71       378
          15       0.91      0.36      0.51       239
          16       0.85      0.98      0.91       499
      

d:\Islington college\masters\nsl-landmark-thesis\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Islington college\masters\nsl-landmark-thesis\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Islington college\masters\nsl-landmark-thesis\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.c